In [0]:
from config import load_config

cfg = load_config("config_databricks.yaml")

In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

fct_applications = spark.table(cfg["fct_applications_table"])

anomaly_hired_before_applied = (
    fct_applications
    .withColumn("apply_date", F.to_date("apply_date"))
    .withColumn("hired_date", F.to_date("hired_date"))
    .filter(
        F.col("hired_date").isNotNull() &
        F.col("apply_date").isNotNull() &
        (F.col("hired_date") < F.col("apply_date"))
    )
)

anomaly_hired_before_applied.write.format("delta").mode("overwrite").saveAsTable(
    cfg["dq_hired_before_applied_table"]
)

In [0]:
from pyspark.sql import functions as F
from config import load_config

cfg = load_config("config_databricks.yaml")

# Load tables
bronze_jobs = spark.table(cfg["bronze_jobs_table"])
bronze_candidates = spark.table(cfg["bronze_candidates_table"])
bronze_education = spark.table(cfg["bronze_education_table"])
bronze_applications = spark.table(cfg["bronze_applications_table"])
bronze_workflow_events = spark.table(cfg["bronze_workflow_events_table"])

dim_job = spark.table(cfg["dim_job_table"])
dim_candidate = spark.table(cfg["dim_candidate_table"])
fct_workflow_events = spark.table(cfg["fct_workflow_events_table"])
fct_applications = spark.table(cfg["fct_applications_table"])

# Helper to standardize check output
def build_check_result(check_name, failed_count, table_name):
    status = "PASS" if failed_count == 0 else "FAIL"
    return [(check_name, table_name, failed_count, status)]

# Duplicate checks
dup_jobs = bronze_jobs.groupBy("job_id").count().filter(F.col("count") > 1).count()
dup_candidates = bronze_candidates.groupBy("candidate_id").count().filter(F.col("count") > 1).count()
dup_applications = bronze_applications.groupBy("application_id").count().filter(F.col("count") > 1).count()
dup_events = bronze_workflow_events.groupBy("event_id").count().filter(F.col("count") > 1).count()

dup_dim_job = dim_job.groupBy("job_id").count().filter(F.col("count") > 1).count()
dup_dim_candidate = dim_candidate.groupBy("candidate_id").count().filter(F.col("count") > 1).count()
dup_fct_applications = fct_applications.groupBy("application_id").count().filter(F.col("count") > 1).count()


# Null checks

null_job_id = bronze_jobs.filter(F.col("job_id").isNull()).count()
null_candidate_id = bronze_candidates.filter(F.col("candidate_id").isNull()).count()
null_application_id = bronze_applications.filter(F.col("application_id").isNull()).count()
null_apply_date = bronze_applications.filter(F.col("apply_date").isNull()).count()
null_event_timestamp = bronze_workflow_events.filter(F.col("event_timestamp").isNull()).count()


# Volume checks

row_count_jobs = bronze_jobs.count()
row_count_candidates = bronze_candidates.count()
row_count_education = bronze_education.count()
row_count_applications = bronze_applications.count()
row_count_events = bronze_workflow_events.count()

volume_results = [
    ("row_count_bronze_jobs", cfg["bronze_jobs_table"], row_count_jobs, "INFO"),
    ("row_count_bronze_candidates", cfg["bronze_candidates_table"], row_count_candidates, "INFO"),
    ("row_count_bronze_education", cfg["bronze_education_table"], row_count_education, "INFO"),
    ("row_count_bronze_applications", cfg["bronze_applications_table"], row_count_applications, "INFO"),
    ("row_count_bronze_workflow_events", cfg["bronze_workflow_events_table"], row_count_events, "INFO"),
]

# Anomaly detection: hired before applied

anomaly_hired_before_applied = (
    fct_applications
    .withColumn("apply_date", F.to_date("apply_date"))
    .withColumn("hired_date", F.to_date("hired_date"))
    .filter(
        F.col("hired_date").isNotNull() &
        F.col("apply_date").isNotNull() &
        (F.col("hired_date") < F.col("apply_date"))
    )
    .select(
        "application_id",
        "job_id",
        "candidate_id",
        "apply_date",
        "hired_date"
    )
)

anomaly_count = anomaly_hired_before_applied.count()

# Save anomaly table
anomaly_hired_before_applied.write.format("delta").mode("overwrite").saveAsTable(
    cfg["dq_hired_before_applied_table"]
)


# Assemble quality results
results = []
results += build_check_result("duplicate_job_id_bronze", dup_jobs, cfg["bronze_jobs_table"])
results += build_check_result("duplicate_candidate_id_bronze", dup_candidates, cfg["bronze_candidates_table"])
results += build_check_result("duplicate_application_id_bronze", dup_applications, cfg["bronze_applications_table"])
results += build_check_result("duplicate_event_id_bronze", dup_events, cfg["bronze_workflow_events_table"])

results += build_check_result("duplicate_job_id_dim", dup_dim_job, cfg["dim_job_table"])
results += build_check_result("duplicate_candidate_id_dim", dup_dim_candidate, cfg["dim_candidate_table"])
results += build_check_result("duplicate_application_id_fact", dup_fct_applications, cfg["fct_applications_table"])

results += build_check_result("null_job_id", null_job_id, cfg["bronze_jobs_table"])
results += build_check_result("null_candidate_id", null_candidate_id, cfg["bronze_candidates_table"])
results += build_check_result("null_application_id", null_application_id, cfg["bronze_applications_table"])
results += build_check_result("null_apply_date", null_apply_date, cfg["bronze_applications_table"])
results += build_check_result("null_event_timestamp", null_event_timestamp, cfg["bronze_workflow_events_table"])

results += build_check_result("hired_before_applied", anomaly_count, cfg["fct_applications_table"])
results += volume_results

dq_results = spark.createDataFrame(
    results,
    schema=["check_name", "table_name", "failed_or_observed_count", "status"]
)

# Save and display

dq_results.write.format("delta").mode("overwrite").saveAsTable(cfg["dq_results_table"])

display(dq_results.orderBy("status", "check_name"))
display(anomaly_hired_before_applied)

In [0]:
spark.sql("SELECT * FROM dq_results ORDER BY status, check_name").show(truncate=False)
spark.sql("SELECT * FROM dq_hired_before_applied LIMIT 20").show()